In [1]:
import pandas as pd

from mpnn import MoleculeFDetector, MoleculeHDetector
from rdkit import Chem
from krfp_models import krfp_models

from tqdm.notebook import tqdm

from ground_truth.highlight_smarts import highlight_atoms_in_mol
from mpnn.validate_whiteboxes import drop_molecule


In [ ]:
for model, _, post_smarts in krfp_models:
    nitrogen_relaxation_post_smarts = post_smarts.replace("[NH]", "[N;H1,H2]")

    positive_df = pd.read_csv(
        f"../artifacts/explainability_method_tester/R2/{model.__class__.__name__}/positive_explainability_results.csv")
    negative_df = pd.read_csv(
        f"../artifacts/explainability_method_tester/R2/{model.__class__.__name__}/negative_explainability_results.csv")

    positive_smiles = positive_df["SMILES"].tolist()
    negative_smiles = negative_df["SMILES"].tolist()

    all_smiles = positive_smiles + negative_smiles

    for smiles in tqdm(all_smiles):
        mol = Chem.MolFromSmiles(smiles)

        gt1 = highlight_atoms_in_mol(mol, post_smarts)
        gt2 = highlight_atoms_in_mol(mol, nitrogen_relaxation_post_smarts)

        assert gt1 == gt2, f"Ground truth mismatch for {smiles}: {gt1} vs {gt2}"


In [ ]:
smiles_pattern_3 = pd.read_parquet("../wombat-smiles/validation_datasets/Pattern 3/validation_molecules.parquet")
smiles_pattern_5 = pd.read_parquet("../wombat-smiles/validation_datasets/Pattern 5/validation_molecules.parquet")

smiles_ds = [smiles_pattern_3["SMILES"].tolist(), smiles_pattern_5["SMILES"].tolist()]

In [ ]:
smarts = [krfp_models[3][2], krfp_models[5][2]]

for smarts_pattern, smiles_list in zip(smarts, smiles_ds):
    nitrogen_relaxation_post_smarts = smarts_pattern.replace("[NH]", "[N;H1,H2]")

    for smiles in tqdm(smiles_list):
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        if drop_molecule(mol):
            continue

        mol = Chem.RemoveAllHs(mol)

        gt1 = highlight_atoms_in_mol(mol, smarts_pattern)
        gt2 = highlight_atoms_in_mol(mol, nitrogen_relaxation_post_smarts)

        assert gt1 == gt2, f"Ground truth mismatch for {smiles}: {gt1} vs {gt2}"